In [2]:
#调用模型
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 定义LLM模型
model = ChatOpenAI(model="gpt-4o-mini", temperature=0, timeout=10,max_tokens=1000)

### 访问上下文
##### 为何重要：当工具能够访问代理状态、运行时上下文以及长期记忆时，它们才最为强大。这使得工具可以做出“有上下文感知”的决策、个性化回应，并在多轮对话中持续保存信息。
##### 工具可通过 ToolRuntime 参数获取运行时信息，该参数提供：
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;State – 随执行流动的可变数据（如消息、计数器、自定义字段）
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;Context – 不可变的配置，如用户 ID、会话详情或应用专属配置
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;Store – 跨对话的持久化长期记忆
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;Stream Writer – 在工具执行时流式输出自定义更新
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;Config – 当前执行的 RunnableConfig
##### &#160;&#160;&#160;&#160;&#160;&#160;&#160;&#160;Tool Call ID – 当前工具调用的 ID
##### 只需在工具签名中加上 runtime: ToolRuntime，即可通过这一个参数访问所有运行时信息；它会自动注入，不会暴露给 LLM。

In [ ]:
from langchain.tools import tool, ToolRuntime

# Access the current conversation state
@tool
def summarize_conversation(
    runtime: ToolRuntime
) -> str:
    """Summarize the conversation so far."""
    messages = runtime.state["messages"]

    human_msgs = sum(1 for m in messages if m.__class__.__name__ == "HumanMessage")
    ai_msgs = sum(1 for m in messages if m.__class__.__name__ == "AIMessage")
    tool_msgs = sum(1 for m in messages if m.__class__.__name__ == "ToolMessage")

    return f"Conversation has {human_msgs} user messages, {ai_msgs} AI responses, and {tool_msgs} tool results"

# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime  # ToolRuntime parameter is not visible to the model
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

##### 这两个工具函数通过 ToolRuntime 访问 LangChain 的内部状态，分别实现了对话条数统计和用户偏好查询功能，且 runtime 参数不会暴露给模型或用户，仅用于函数内部逻辑。

### 更新状态
##### 使用 Command 更新代理的状态或控制图的执行流程：

In [ ]:
from langgraph.types import Command
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langchain.tools import tool, ToolRuntime

# Update the conversation history by removing all messages
@tool
def clear_conversation() -> Command:
    """Clear the conversation history."""

    return Command(
        update={
            "messages": [RemoveMessage(id=REMOVE_ALL_MESSAGES)],
        }
    )

# Update the user_name in the agent state
@tool
def update_user_name(
    new_name: str,
    runtime: ToolRuntime
) -> Command:
    """Update the user's name."""
    return Command(update={"user_name": new_name})

#### PS：可以这么理解，我跟ai对话时，我先说我是小红，于是graph便记录了user_name这么一个参数为小红，如果无法改变图状态，我后面跟ai说，其实我是小蓝，那么记录的user_name不改变，ai只能通过记录，“口头上改变”；而如果可以改变图状态，我后面跟ai说，其实我是小蓝，那么user_name将从小红变为小蓝。
#### “口头上改变”并不可靠，如果后续记录中，ai遗忘这条消息或者其他情况，user_name并没有改变，ai仍会称呼用户为小红；可直接修改状态，则不会有这种状况。

### Context
##### 通过 runtime.context 访问不可变的配置与上下文数据，例如用户 ID、会话详情或应用专属配置。
##### 工具可以通过以下方式访问运行时上下文：ToolRuntime

In [12]:
from dataclasses import dataclass
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


USER_DATABASE = {
    "user123": {
        "name": "Alice Johnson",
        "account_type": "Premium",
        "balance": 5000,
        "email": "alice@example.com"
    },
    "user456": {
        "name": "Bob Smith",
        "account_type": "Standard",
        "balance": 1200,
        "email": "bob@example.com"
    }
}

@dataclass
class UserContext:
    user_id: str

@tool
def get_account_info(runtime: ToolRuntime[UserContext]) -> str:
    """Get the current user's account information."""
    user_id = runtime.context.user_id

    if user_id in USER_DATABASE:
        user = USER_DATABASE[user_id]
        return f"Account holder: {user['name']}\nType: {user['account_type']}\nBalance: ${user['balance']}"
    return "User not found"

agent = create_agent(
    model,
    tools=[get_account_info],
    context_schema=UserContext,
    system_prompt="You are a financial assistant."
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's my current balance?"}]},
    context=UserContext(user_id="user123")
)

print(result['messages'][3].content)

Your current balance is $5,000.


#### PS：runtime.context 里面能放什么，完全由把工具塞进图的那个人（写应用代码的你）决定。它只是一层只读字典
#### PS：runtime.context 里的值不是写死在源代码里，而是每次调用图的时候由“调用者”传进来；传一次就固定，“工具运行期间改不了”。
#### PS：如果某个工具在运行途中对state进行了修改，那么就能修改成功，且在后续运行途中，调用的就是新的修改成功后的数据；而context是无法在运行过程中进行修改的，任何方法都不可以
#### PS：context的生命周期是每一次invoke，而state是整个对话记录

### Memory(Store)
##### 使用存储跨对话访问持久性数据。runtime.store可通过以下方式访问，并允许您保存和检索特定于用户或特定于应用程序的数据。

In [20]:
from typing import Any
from langgraph.store.memory import InMemoryStore
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


# Access memory
@tool
def get_user_info(user_id: str, runtime: ToolRuntime) -> str:
    """Look up user info."""
    store = runtime.store
    user_info = store.get(("users",), user_id)
    return str(user_info.value) if user_info else "Unknown user"

# Update memory
@tool
def save_user_info(user_id: str, user_info: dict[str, Any], runtime: ToolRuntime) -> str:
    """Save user info."""
    store = runtime.store
    store.put(("users",), user_id, user_info)
    return "Successfully saved user info."

store = InMemoryStore()
agent = create_agent(
    model,
    tools=[get_user_info, save_user_info],
    store=store
)

# First session: save user info
agent.invoke({
    "messages": [{"role": "user", "content": "Save the following user: userid: abc123, name: Foo, age: 25, email: foo@langchain.dev"}]
})

# Second session: get user info
agent.invoke({
    "messages": [{"role": "user", "content": "Get user info for user with id 'abc123'"}]
})
# Here is the user info for user with ID "abc123":
# - Name: Foo
# - Age: 25
# - Email: foo@langchain.dev

GraphRecursionError: Recursion limit of 100 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT

### 流写入器
##### 使用runtime.stream_writer时从工具执行自定义更新。这对于向用户提供有关工具正在执行的作的实时反馈非常有用。

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_weather(city: str, runtime: ToolRuntime) -> str:
    """Get weather for a given city."""
    writer = runtime.stream_writer

    # Stream custom updates as the tool executes
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")

    return f"It's always sunny in {city}!"